# Second draws --- does block34 reproduce? (2026-08-29)

The sweep has five arms and, since today, a **measured** noise floor. The floor
is the point of this notebook: two independent unseeded draws of the `frozen`
specification --- same data, same folds, same seed, nothing changed --- differ
by **+0.0035** in the paired sixteen-station mean at **t = +1.34**, with one
station moving 0.0256.

A comparison of one specification against *itself* produced most of a
conventional significance threshold. So the paper cannot yet say what
block34's +0.0109 at t = +2.45 means, because block34 was run once.

This notebook now runs **one arm**, `block34_rep2` --- the decisive one. Is
block34's $+0.0109$ the model or the draw?

The other two replicates moved to the local machine on 2026-08-30, and the
reason is measured rather than guessed. They train a frozen trunk on the cached
features, so they need no GPU, and the local machine is three times faster at
it: **7.2 min/fold against 20.8 here**, taken from the two runs' own manifests.
`frozen_rep3` belongs there for a second reason that is about the science --- the
first two draws of that specification were local, so a third local draw measures
training nondeterminism with the machine held fixed, where running it here would
confound platform with draw.

Only the unfrozen arm has to be on a GPU: unfreezing reads the image pack
instead of the cache and is roughly eight times slower per epoch. Expect about
25 min/fold on a T4, so around 6.5 hours for sixteen.

**Check the runtime is a GPU before starting** --- Runtime, Change runtime type,
T4 GPU. On a CPU runtime this arm will not finish.

**Safe to interrupt.** Every fold syncs to Drive as it completes and re-running
the run cell picks up from whatever is already there, so closing the lid costs
the fold in flight and nothing before it. Set `REP_ARMS=all` in the run cell if
you ever want all three here.


In [ ]:
from google.colab import drive; drive.mount('/content/drive')
!rm -rf /content/repo && git clone -q -b v13-honest-labels https://github.com/Mo119m/primates-sound-detection /content/repo
%cd /content/repo
!git log --oneline -1
U = '/content/drive/MyDrive/primates-sound-detection'
!mkdir -p /content/dataF && cp -n {U}/v13_images.npy {U}/v13_index.csv {U}/manifest.csv /content/dataF/
!ls -la /content/dataF

In [ ]:
# Is the Drive copy the same bytes as the build every number in the paper
# comes from? Row count is not enough -- two builds can share a row count and
# differ in content, and Drive is a copy made on 2026-08-20 that nothing has
# re-verified since. These three digests were taken from the local
# full_2026-08-19 artifacts, whose mtimes are all 2026-08-19, i.e. before the
# upload and unchanged since. e79cfaee6813 is the index sha every run.json in
# the shipped sweep records as its input.
import hashlib
import pandas as pd

EXPECT = {
    'v13_index.csv':  'e79cfaee6813075553f8d4878ea54593b093a114e0f8456efe728e16a1f92ab5',
    'manifest.csv':   'a93f43d4d82115abfa1d6399f0756672b66bbadd69855ddbc402edca45c89dd2',
    'v13_images.npy': 'af9827632060ab8afe5b97f7f0b2034442c1d812c0dcdbaa5c80301238debb81',
}
for name, want in EXPECT.items():
    h = hashlib.sha256()
    with open(f'/content/dataF/{name}', 'rb') as fh:
        for chunk in iter(lambda: fh.read(1 << 24), b''):
            h.update(chunk)
    got = h.hexdigest()
    print(f'{name:16s} {got[:12]}  {"OK" if got == want else "MISMATCH"}')
    assert got == want, (
        f'{name} on Drive is not the build this paper reports. Re-upload it '
        f'from data/outputs/v13_runs/full_2026-08-19/ before training.')

i = pd.read_csv('/content/dataF/v13_index.csv')
assert len(i) == 22169, 'wrong dataset on Drive'
print(len(i), 'rows;', i.label.value_counts().to_dict())
print('Drive artifacts are byte-identical to the shipped build')

In [ ]:
# The time gate needs its lookup table. Without it every fold trains fine and
# comes out missing the four columns the comparison reads -- which is what
# happened on 24 August and cost seven folds. Fail here, where it is free.
import os
import pandas as pd
G = '/content/repo/data/outputs/auto_cleanup/review_gate_table.csv'
assert os.path.exists(G), 'no review gate table in the clone -- pull the branch again'
g = pd.read_csv(G)
print(len(g), 'rows,', list(g.columns))
assert len(g) == 6189, 'wrong gate table'
assert set(g.columns) == {'file', 'timestamp', 'start_s'}, 'unexpected columns'
print('time gate has its clock')

In [ ]:
A = '--manifest /content/dataF/manifest.csv --index /content/dataF/v13_index.csv --images /content/dataF/v13_images.npy --cache /content/dataF/v13_features.npy'
!python scripts/train_v13_loso.py --prepare-cache-only --overwrite {A} --out /content/warm.csv --run-metadata /content/cacheF.run.json

In [ ]:
# Sixteen folds, synced fold by fold. Re-run this cell after any
# disconnect: finished folds in Drive are skipped, so a dead session costs
# the fold in flight and nothing before it.
#
# REP_ARMS defaults to block34_rep2 -- the other two replicates run locally,
# where the same work is three times faster and does not need a GPU.
!python colab/run_replicates.py

In [ ]:
# Read the arms back without retraining, once they exist. Each replicate is
# printed beside its own first draw; the draw-to-draw line is the one that
# decides whether the original measured the model or the draw.
import sys
sys.argv = ['x']
sys.path.insert(0, '/content/repo/colab')
import run_replicates
run_replicates.summarise()